# Portfolio DataFrames Explorer

Interactive notebook for exploring portfolio data using the same `data.py` and `service.py`
layers that power the portfolio_app API. All DataFrames are produced from MySQL queries
and service-layer transformations.

In [ ]:
# Setup: add portfolio_app/src to path and load environment
import sys
import os

# Windows encoding fix (DEVELOPMENT_RULES §2)
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

from pathlib import Path

# Derive paths relative to this notebook (analysis/ -> portfolio_app/ -> project root)
_notebook_dir = Path(os.getcwd())               # notebook kernel CWD
_portfolio_app = _notebook_dir.parent            # portfolio_app/
project_root = _portfolio_app.parent             # project root
src_dir = _portfolio_app / "src"

# Fallback: if CWD isn't the analysis dir, try __file__
if not (src_dir / "data.py").exists():
    src_dir = Path(r"I:\masterswork\git\OpenBB\portfolio_app\src")
    project_root = Path(r"I:\masterswork\git\OpenBB")

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Skip 67-table auto-create overhead (DEVELOPMENT_RULES §3)
os.environ.setdefault("FMP_CACHE_AUTO_CREATE_DB", "false")

# Load .env for database credentials
from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print(f"Project root: {project_root}")
print(f"Source dir:    {src_dir}")
print("✅ Environment loaded")

✅ Environment loaded


In [7]:
# Import data and service layers
from data import (
    get_positions_df,
    get_all_snapshots_df,
    get_espp_df,
    get_latest_prices_df,
    get_equity_historical_df,
    get_distinct_symbols,
    get_distinct_accounts,
    get_distinct_owners,
    check_db,
)
import service

# Verify DB connection
db_ok = check_db()
print(f"Database connected: {db_ok}")

Database connected: True


## 1. Raw Positions DataFrame
All position lots from the latest snapshot, joined with account owner info.

In [8]:
# Fetch raw positions (latest snapshot)
positions_raw = get_positions_df()
print(f"Shape: {positions_raw.shape}")
print(f"Columns: {list(positions_raw.columns)}")
print(f"Unique symbols: {positions_raw['symbol'].nunique()}")
print(f"Unique accounts: {positions_raw['account_name'].nunique()}")
print()
positions_raw.head(10)

Shape: (584, 16)
Columns: ['account_name', 'owner', 'symbol', 'description', 'quantity', 'avg_cost_basis', 'cost_basis_total', 'current_value', 'total_gain_loss', 'pct_gain_loss', 'term', 'acquired', 'share_source', 'snapshot_date', 'grant_date', 'transfer_avail_date']
Unique symbols: 93
Unique accounts: 8



,account_name,owner,symbol,description,quantity,avg_cost_basis,cost_basis_total,current_value,total_gain_loss,pct_gain_loss,term,acquired,share_source,snapshot_date,grant_date,transfer_avail_date
0,Individual - TOD (X65957336),Prashant,Cash,HELD IN MONEY MARKET,0.00,0.00,0.00,"18,843.08",0.00,0.00,,NaN,,2026-02-20T01:39:00,NaN,NaN
1,Individual - TOD (X65957336),Prashant,04599D597,ASIA BROADBAND INC $0.001 NEVADA RESTRICTED,0.00,0.00,0.00,0.00,0.00,0.00,,NaN,,2026-02-20T01:39:00,NaN,NaN
2,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,40.00,132.97,"5,318.73","10,423.20","5,104.47",95.97,Long,2022-06-13,,2026-02-20T01:39:00,NaN,NaN
3,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,10.00,132.84,"1,328.40","2,605.80","1,277.40",96.16,Long,2022-06-13,,2026-02-20T01:39:00,NaN,NaN
4,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,10.00,145.41,"1,454.09","2,605.80","1,151.71",79.20,Long,2022-06-03,,2026-02-20T01:39:00,NaN,NaN
5,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,20.00,150.13,"3,002.70","5,211.60","2,208.90",73.56,Long,2022-06-02,,2026-02-20T01:39:00,NaN,NaN
6,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,10.00,140.00,"1,400.00","2,605.80","1,205.80",86.13,Long,2022-05-24,,2026-02-20T01:39:00,NaN,NaN
7,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,5.00,160.00,800.00,"1,302.90",502.90,62.86,Long,2022-03-07,,2026-02-20T01:39:00,NaN,NaN
8,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,5.00,160.00,800.00,"1,302.90",502.90,62.86,Long,2022-02-23,,2026-02-20T01:39:00,NaN,NaN
9,Individual - TOD (X65957336),Prashant,AAPL,APPLE INC,20.00,141.69,"2,833.90","5,211.60","2,377.70",83.90,Long,2021-10-12,,2026-02-20T01:39:00,NaN,NaN


## 2. Refresh Market Values
Fetch latest prices from the cache and recalculate current_value, gain/loss.

In [ ]:
# Reload provider + data modules to pick up source changes (as_of_date param)
import importlib
import openbb_fmp_cached.models.equity_historical as _eh_mod
importlib.reload(_eh_mod)

import data as _data_mod
importlib.reload(_data_mod)
from data import get_latest_prices_df  # re-import after reload

from datetime import date, timedelta
yesterday = date.today() - timedelta(days=1)

# Get latest prices for all portfolio symbols as of yesterday
symbols = positions_raw["symbol"].unique().tolist()
print(f"Portfolio contains {len(symbols)} unique symbols\n")

prices = get_latest_prices_df(symbols, as_of_date=yesterday, verbose=True)

print(f"\nResult: {len(prices)} rows, columns={list(prices.columns)}")
prices.head()

Portfolio contains 93 unique symbols

Fetching latest prices for 93 symbols (as of 2026-02-23) ...
  [1/93] Cash     → $     92.80  (2026-02-23)  [10.9s]


No data found for 04599D597.
I:\masterswork\git\OpenBB\openbb_platform\providers\fmp_cached\openbb_fmp_cached\models\equity_historical.py:1044: UserWarning: No data found for 04599D597.
  warn(msg)
Gap fetch/store failed for 04599D597 (2026-02-09→2026-02-23): No data found for 04599D597.


  [2/93] 04599D597 → NO DATA  [8.8s]
  [3/93] AAPL     → $    266.18  (2026-02-23)  [4.1s]
  [4/93] AAPT     → $      0.00  (2026-02-23)  [4.1s]
  [5/93] ABNB     → $    122.96  (2026-02-23)  [4.1s]
  [6/93] AGNC     → $     11.27  (2026-02-23)  [4.1s]
  [7/93] AMT      → $    190.05  (2026-02-23)  [4.0s]
  [8/93] AMZN     → $    205.27  (2026-02-23)  [4.1s]
  [9/93] BAC      → $     51.07  (2026-02-23)  [4.1s]
  [10/93] CMG      → $     36.61  (2026-02-23)  [4.1s]
  [11/93] CRM      → $    178.16  (2026-02-23)  [4.1s]
  [12/93] FSKAX    → $    187.42  (2026-02-23)  [4.1s]
  [13/93] GOOGL    → $    311.49  (2026-02-23)  [4.1s]
  [14/93] HIMS     → $     15.51  (2026-02-23)  [4.1s]
  [15/93] META     → $    637.25  (2026-02-23)  [4.1s]
  [16/93] MSFT     → $    384.47  (2026-02-23)  [4.1s]
  [17/93] NOW      → $    100.80  (2026-02-23)  [4.1s]
  [18/93] NVDA     → $    191.55  (2026-02-23)  [4.1s]
  [19/93] PTON     → $      4.13  (2026-02-23)  [4.1s]
  [20/93] QQQ      → $    601.41  (

No data found for NHX202764.
I:\masterswork\git\OpenBB\openbb_platform\providers\fmp_cached\openbb_fmp_cached\models\equity_historical.py:1044: UserWarning: No data found for NHX202764.
  warn(msg)
Gap fetch/store failed for NHX202764 (2026-02-09→2026-02-23): No data found for NHX202764.


  [32/93] NHX202764 → NO DATA  [8.9s]


...

  [90/93] 36320A203 → NO DATA  [8.9s]
  [91/93] FBIFX    → $     29.88  (2026-02-23)  [10.9s]
  [92/93] SCHB     → $     26.32  (2026-02-23)  [11.0s]


No data found for 09261F598.
I:\masterswork\git\OpenBB\openbb_platform\providers\fmp_cached\openbb_fmp_cached\models\equity_historical.py:1044: UserWarning: No data found for 09261F598.
  warn(msg)
Gap fetch/store failed for 09261F598 (2026-02-09→2026-02-23): No data found for 09261F598.


  [93/93] 09261F598 → NO DATA  [8.9s]
Done — 87/93 prices fetched in 668.6s

Result: 87 rows, columns=['symbol', 'close', 'price_date']


,symbol,close,price_date
0,Cash,92.80,2026-02-23
1,AAPL,266.18,2026-02-23
2,AAPT,0.00,2026-02-23
3,ABNB,122.96,2026-02-23
4,AGNC,11.27,2026-02-23


: 

In [ ]:
# Refresh positions with live market prices
positions = service.refresh_market_values(positions_raw, prices)
print(f"Positions refreshed: {len(positions)} lots")
positions[["symbol", "account_name", "quantity", "avg_cost_basis",
           "cost_basis_total", "current_value", "total_gain_loss",
           "pct_gain_loss", "price_date"]].head(10)

## 3. Portfolio Summary (by Symbol)
Aggregated view: total quantity, cost basis, current value, and return % per ticker.

In [ ]:
summary = service.summary_by_symbol(positions)
print(f"Summary: {len(summary)} symbols")
print(f"Total portfolio value: ${summary['total_current_value'].sum():,.2f}")
print(f"Total cost basis:      ${summary['total_cost_basis'].sum():,.2f}")
print(f"Total gain/loss:       ${summary['total_gain_loss'].sum():,.2f}")
print()
summary

## 4. Account Allocation
How the portfolio is distributed across brokerage accounts.

In [ ]:
allocation = service.allocation_by_account(positions)
print(f"Accounts: {len(allocation)}")
allocation

## 5. Performance Ranking
Top gainers and losers by percent return.

In [ ]:
perf = service.performance_ranking(positions)
print("=== Top 5 Gainers ===")
display(perf.head(5))
print("\n=== Top 5 Losers ===")
display(perf.tail(5))

## 6. Cost Basis Lots
Per-lot detail with short/long-term classification — useful for tax planning.

In [ ]:
cost_basis = service.cost_basis_lots(positions)
print(f"Total lots: {len(cost_basis)}")
print(f"Short-term lots: {(cost_basis['term'] == 'Short-Term').sum()}")
print(f"Long-term lots:  {(cost_basis['term'] == 'Long-Term').sum()}")
print()
cost_basis.head(10)

## 7. Tax Summary
Short-term vs long-term gains/losses grouped by account.

In [ ]:
tax = service.tax_summary(positions)
print("Tax Summary by Account and Term:")
tax

## 8. Snapshot History
Historical snapshots — track portfolio value over time.

In [ ]:
snapshots_raw = get_all_snapshots_df()
snapshots = service.snapshot_totals(snapshots_raw)
print(f"Snapshots: {len(snapshots)} dates")
snapshots

## 9. ESPP Purchases
Employee Stock Purchase Plan history with discount analysis.

In [ ]:
espp = get_espp_df()
if not espp.empty:
    print(f"ESPP purchases: {len(espp)}")
    print(f"Total invested:  ${espp['purchase_value'].sum():,.2f}")
    print(f"Avg discount:    {espp['discount_pct'].mean():.1f}%")
    print()
    display(espp.head(10))
else:
    print("No ESPP data found.")

## 10. Available Filters
Dropdown values available for filtering widgets.

In [ ]:
print("Symbols:", [d['value'] for d in get_distinct_symbols()])
print("Accounts:", [d['value'] for d in get_distinct_accounts()])
print("Owners:", [d['value'] for d in get_distinct_owners()])

## 11. Filtered Views
Examples of filtering by owner, account, or symbol.

In [ ]:
# Filter by a specific owner (change the name to match your data)
owners = get_distinct_owners()
if owners:
    first_owner = owners[0]["value"]
    print(f"Filtering summary for owner: {first_owner}")
    owner_summary = service.summary_by_symbol(positions, owner=first_owner)
    display(owner_summary)
else:
    print("No owners found.")

In [ ]:
# Filter cost basis for a specific symbol
symbols_list = get_distinct_symbols()
if symbols_list:
    first_sym = symbols_list[0]["value"]
    print(f"Cost basis lots for: {first_sym}")
    sym_lots = service.cost_basis_lots(positions, symbol=first_sym)
    display(sym_lots)
else:
    print("No symbols found.")

## 12. Quick Stats
Portfolio-level statistics at a glance.

In [ ]:
if not summary.empty:
    total_value = summary["total_current_value"].sum()
    total_cost = summary["total_cost_basis"].sum()
    total_gain = summary["total_gain_loss"].sum()
    pct = (total_gain / total_cost * 100) if total_cost else 0

    stats = pd.DataFrame({
        "Metric": [
            "Total Stocks",
            "Total Lots",
            "Total Accounts",
            "Total Cost Basis",
            "Total Current Value",
            "Total Gain/Loss",
            "Overall Return %",
            "Best Performer",
            "Worst Performer",
        ],
        "Value": [
            f"{summary['symbol'].nunique()}",
            f"{len(positions)}",
            f"{allocation['account_name'].nunique()}",
            f"${total_cost:,.2f}",
            f"${total_value:,.2f}",
            f"${total_gain:,.2f}",
            f"{pct:.2f}%",
            f"{perf.iloc[0]['symbol']} ({perf.iloc[0]['pct_return']:.1f}%)" if not perf.empty else "N/A",
            f"{perf.iloc[-1]['symbol']} ({perf.iloc[-1]['pct_return']:.1f}%)" if not perf.empty else "N/A",
        ]
    })
    display(stats)
else:
    print("No portfolio data available.")

## 13. Stats by Account
Value, cost basis, gain/loss, return %, top/worst performer, and lot counts per brokerage account.

In [ ]:
accounts = [d["value"] for d in get_distinct_accounts()]

for acct in accounts:
    acct_positions = positions[positions["account_name"] == acct]
    if acct_positions.empty:
        continue

    acct_summary = service.summary_by_symbol(positions, account=acct)
    acct_cost_basis = service.cost_basis_lots(positions, account=acct)

    total_value = acct_summary["total_current_value"].sum()
    total_cost = acct_summary["total_cost_basis"].sum()
    total_gain = acct_summary["total_gain_loss"].sum()
    pct = (total_gain / total_cost * 100) if total_cost else 0

    best = acct_summary.sort_values("pct_return", ascending=False).iloc[0] if len(acct_summary) > 0 else None
    worst = acct_summary.sort_values("pct_return", ascending=True).iloc[0] if len(acct_summary) > 0 else None

    st_lots = (acct_cost_basis["term"] == "Short-Term").sum() if "term" in acct_cost_basis.columns else 0
    lt_lots = (acct_cost_basis["term"] == "Long-Term").sum() if "term" in acct_cost_basis.columns else 0

    print(f"\n{'='*60}")
    print(f"  Account: {acct}")
    print(f"{'='*60}")

    acct_stats = pd.DataFrame({
        "Metric": [
            "Symbols Held",
            "Total Lots",
            "Short-Term Lots",
            "Long-Term Lots",
            "Total Cost Basis",
            "Total Current Value",
            "Total Gain/Loss",
            "Return %",
            "Best Performer",
            "Worst Performer",
        ],
        "Value": [
            f"{acct_summary['symbol'].nunique()}",
            f"{len(acct_cost_basis)}",
            f"{st_lots}",
            f"{lt_lots}",
            f"${total_cost:,.2f}",
            f"${total_value:,.2f}",
            f"${total_gain:,.2f}",
            f"{pct:.2f}%",
            f"{best['symbol']} ({best['pct_return']:.1f}%)" if best is not None else "N/A",
            f"{worst['symbol']} ({worst['pct_return']:.1f}%)" if worst is not None else "N/A",
        ]
    })
    display(acct_stats)
    print(f"\nHoldings in {acct}:")
    display(acct_summary)

NameError: name 'positions' is not defined

## 14. Stats by Owner
Value, cost basis, gain/loss, return %, allocation spread, and performance per portfolio owner.

In [ ]:
owner_list = [d["value"] for d in get_distinct_owners()]

for owner in owner_list:
    owner_summary = service.summary_by_symbol(positions, owner=owner)
    owner_alloc = service.allocation_by_account(positions, owner=owner)
    owner_perf = service.performance_ranking(positions, owner=owner)
    owner_tax = service.tax_summary(positions, owner=owner)

    if owner_summary.empty:
        continue

    total_value = owner_summary["total_current_value"].sum()
    total_cost = owner_summary["total_cost_basis"].sum()
    total_gain = owner_summary["total_gain_loss"].sum()
    pct = (total_gain / total_cost * 100) if total_cost else 0

    best = owner_perf.iloc[0] if not owner_perf.empty else None
    worst = owner_perf.iloc[-1] if not owner_perf.empty else None

    st_gain = owner_tax.loc[owner_tax["term"] == "Short-Term", "total_gain_loss"].sum() if not owner_tax.empty else 0
    lt_gain = owner_tax.loc[owner_tax["term"] == "Long-Term", "total_gain_loss"].sum() if not owner_tax.empty else 0

    print(f"\n{'='*60}")
    print(f"  Owner: {owner}")
    print(f"{'='*60}")

    owner_stats = pd.DataFrame({
        "Metric": [
            "Unique Symbols",
            "Accounts",
            "Total Cost Basis",
            "Total Current Value",
            "Total Gain/Loss",
            "Return %",
            "Short-Term Gain/Loss",
            "Long-Term Gain/Loss",
            "Best Performer",
            "Worst Performer",
        ],
        "Value": [
            f"{owner_summary['symbol'].nunique()}",
            f"{owner_alloc['account_name'].nunique()}",
            f"${total_cost:,.2f}",
            f"${total_value:,.2f}",
            f"${total_gain:,.2f}",
            f"{pct:.2f}%",
            f"${st_gain:,.2f}",
            f"${lt_gain:,.2f}",
            f"{best['symbol']} ({best['pct_return']:.1f}%)" if best is not None else "N/A",
            f"{worst['symbol']} ({worst['pct_return']:.1f}%)" if worst is not None else "N/A",
        ]
    })
    display(owner_stats)

    print(f"\nAccount Allocation for {owner}:")
    display(owner_alloc)

    print(f"\nTop Holdings for {owner}:")
    display(owner_summary.head(10))